# Activation screening and radiological totals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nukehub-dev/nucleide/blob/main/notebooks/activation-clearance.ipynb)

Parse an ALARA activation-output listing, screen the inventory with the sum-of-fractions rule, and fold the Sublet radiological totals — committed hazards, transport ratio, IAEA clearance index, and gamma dose. Dose coefficients, limits, levels, and attenuation data are caller inputs throughout, never vendored.

> On Colab, run `%pip install "nucleide>=0.16.0"` first. Coefficient values below are synthetic.

In [ ]:
import nucleide.alara as alara

# Parse the committed upstream ALARA listing (2838 rows).
with open("fixtures/alara/output/sample2.out") as handle:
    rows = alara.alara_parse_output(handle.read(), "sample2")
print(f"{len(rows)} rows, e.g. {rows[0]['nuclide']} = {rows[0]['value']:.4g} {rows[0]['var_unit']}")
assert len(rows) == 2838

# EU Annex VII Table A default (Bq/g, transcribed from the directive).
eu = alara.alara_clearance_eu_table()
print(f"{len(eu)} entries; Co-60 limit = {eu['Co-60']} Bq/g")
assert eu["Co-60"] == 0.1

## Screening and hazards over a caller inventory

Activities and the limit basis must share one unit basis. The ingestion/inhalation coefficients are caller-supplied 50-year committed values (Sv/Bq).

In [ ]:
inv = {"Co60": 10.0, "H3": 5.0, "Fe55": 4.0}
limits = {"Co60": 10.0, "H3": 5.0, "Fe55": 2.0}
print("clearance index:", alara.alara_clearance_index(inv, limits))
sof = alara.alara_sum_of_fractions(inv, limits)
print("sum:", sof["sum"], "class:", sof["class"], "dominant:", sof["max_nuclide"])
assert sof["sum"] == 4.0

ing = alara.alara_ingestion_hazard(
    [
        {"nuclide": "Co60", "activity_bq": 10.0, "e_ing_sv_per_bq": 3.0},
        {"nuclide": "H3", "activity_bq": 5.0, "e_ing_sv_per_bq": 2.0},
    ]
)
print(f"ingestion: {ing['total_sv']} Sv total, {ing['ex_tritium_sv']} Sv ex-tritium")
assert ing["total_sv"] == 40.0

## Transport ratio and IAEA clearance variant

In [ ]:
tra = alara.alara_transport_ratio(
    [
        {"nuclide": "Co60", "activity_bq": 1e12, "a2_tbq": 1.0},
        {"nuclide": "H3", "activity_bq": 2e12, "a2_tbq": 1.0},
    ]
)
print(f"Bq/A2 = {tra['ratio']}, effective A2 = {tra['effective_a2_tbq']} TBq")
assert tra["ratio"] == 3.0

iaea = alara.alara_iaea_clearance_index(
    2.0,
    [
        {"nuclide": "Co60", "activity_bq": 10.0, "limit_bq_per_kg": 10.0},
        {"nuclide": "H3", "activity_bq": 5.0, "limit_bq_per_kg": 5.0},
    ],
)
print(f"IAEA index = {iaea['index']}, class = {iaea['clearance_class']}")
assert iaea["index"] == 1.0 and iaea["clearance_class"] == "satisfied"